# Chapter 4 &mdash; The Pumping Lemma in Predicate Logic

**Concept 22 of the Chapter 4 decomposition:** *The Pumping Lemma in Predicate Logic, and a More General Version*

The quantified form &mdash; and an SMT game in which the solver, not you, answers "for all splits".

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Pumping-Lemma-Predicate-Logic/Concept-Pumping-Lemma-Predicate-Logic.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]



import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$$Reg(L) \Rightarrow \exists N : \forall w \in L : [\,|w|\ge N \Rightarrow \exists x,y,z :
w=xyz \wedge |xy|\le N \wedge y\ne\varepsilon \wedge \forall i\ge0 : xy^iz\in L\,]$$

Negated, every quantifier flips:

$$\neg Cond(L) \equiv \forall N : \exists w \in L : |w|\ge N \wedge \forall x,y,z :
[\,w=xyz \wedge |xy|\le N \wedge y\ne\varepsilon \Rightarrow \exists i : xy^iz\notin L\,]$$

Read off who chooses what: the **adversary** picks $N$ and the **split**; **you** pick
$w$ and $i$.

**And there is the trouble with doing this by hand.** That $\forall x,y,z$ is the step
everyone fakes. Try three splits, watch them break, declare victory &mdash; when a
*regular* language would have survived those same three. Worked examples make it
worse: they hand you splits that happen to break, and you learn the ritual instead of
the argument.

So there are no worked splits in this notebook. The quantifier goes to a **solver**
instead. A split becomes a symbolic $(x,y,z)$ and z3 answers for all of them at once.
**UNSAT** means no split survives &mdash; and nothing can be cherry-picked, because
nothing was picked.

## 2. Definitions

### The solver, and a language as a predicate

In [ ]:
# --- the solver ----------------------------------------------------------
try:
    import z3
except ImportError:
    import subprocess, sys
    print("installing z3 ...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'])
    import z3
from z3 import (And, Concat, Contains, If, IndexOf, InRe, IntVal, Length,
                Re, Solver, Star, StringVal, Strings, sat, unsat)
print("z3", z3.get_version_string())


# --- a language is a PREDICATE over a z3 String --------------------------
# Anything writable this way, the solver can reason about -- including
# languages that are NOT regular, hence not expressible as a regex.

def zeros_then_ones(f):
    # Build a predicate on 0*1* from a condition f(#0s, #1s).
    #
    # The shape 0*1* IS a regex, so z3 takes it directly.  Under that shape
    # the count of 0s is the index of the first 1, so counting is a
    # subtraction -- and that is how a NON-regular condition such as
    # '#0 equals #1' gets expressed in a theory whose regexes could never
    # state it.
    def P(s):
        shape = InRe(s, Concat(Star(Re(StringVal("0"))),
                               Star(Re(StringVal("1")))))
        i = IndexOf(s, StringVal("1"), IntVal(0))
        a = If(i == -1, Length(s), i)
        return And(shape, f(a, Length(s) - a))
    return P


P_anbn = zeros_then_ones(lambda a, b: a == b)        # 0^n 1^n -- the stock one
P_lt   = zeros_then_ones(lambda a, b: a < b)         # 0^n 1^m, n < m
P_2n   = zeros_then_ones(lambda a, b: b == 2 * a)    # 0^n 1^2n

Y_SHAPES = {
    'all 0s':    lambda y: InRe(y, Star(Re(StringVal("0")))),
    'all 1s':    lambda y: InRe(y, Star(Re(StringVal("1")))),
    'straddles': lambda y: And(Contains(y, StringVal("0")),
                               Contains(y, StringVal("1"))),
}

### The game

In [ ]:
class PumpGame:
    # Refute Cond(L), with the solver answering the quantifier you cannot.
    #
    # The step everyone fakes is "for ALL splits".  Enumerate a few by hand
    # and nothing is proved -- pick convenient ones and any language looks
    # non-regular.  Here a split is a SYMBOLIC (x, y, z) and the solver
    # answers the forall in one shot.  There is no example to cherry-pick.

    TIMEOUT_MS = 10000

    def __init__(self, P, name):
        self.P, self.name, self.N, self.w = P, name, None, None

    # -- the solver call, three-valued ---------------------------------
    def _ask(self, extra=None, pumps=None):
        # Returns (status, witness); status is 'sat', 'unsat' or 'unknown'.
        #
        # 'unknown' is NOT 'unsat'.  A solver that gave up has proved
        # nothing, and reporting it as "no split survives" would manufacture
        # a proof out of a timeout -- the very sin this notebook exists to
        # prevent.
        x, y, z = Strings('x y z')
        s = Solver()
        s.set('timeout', self.TIMEOUT_MS)
        s.add(Concat(x, y, z) == StringVal(self.w))
        s.add(Length(x) + Length(y) <= self.N, Length(y) >= 1)
        if extra is not None:
            s.add(extra(y))
        for i in (pumps or []):
            s.add(self.P(Concat(x, *([y] * i), z)) if i else self.P(Concat(x, z)))
        r = s.check()
        if r != sat:
            return ('unsat' if r == unsat else 'unknown'), None
        m = s.model()
        g = lambda v: m[v].as_string() if m[v] is not None else ''
        return 'sat', (g(x), g(y), g(z))

    def _member(self, w):
        s = Solver(); s.set('timeout', self.TIMEOUT_MS)
        s.add(self.P(StringVal(w)))
        return s.check() == sat

    # -- the adversary moves first -------------------------------------
    def adversary(self, N):
        self.N, self.w = N, None
        print("Adversary: 'Suppose %s is regular. Then it has some pumping"
              % self.name)
        print("            constant; I choose N = %d.'" % N)
        print("You: choose w in %s with |w| >= %d  ->  choose_w(w)"
              % (self.name, N))
        return self

    # -- you choose w ---------------------------------------------------
    def choose_w(self, w):
        if self.N is None:
            print("adversary(N) first."); return self
        if not self._member(w):
            print("REJECTED: %r is not in %s. The lemma constrains only"
                  " members." % (w, self.name)); return self
        if len(w) < self.N:
            print("REJECTED: |w| = %d < N = %d. The lemma says nothing about"
                  " strings shorter than N." % (len(w), self.N)); return self
        self.w = w
        print("Accepted: w = %r   (in %s, |w| = %d >= N = %d)"
              % (w, self.name, len(w), self.N))
        print("Next: which shapes can y take?  ->  y_shapes([...])")
        return self

    # -- you enumerate the cases; the solver marks your work ------------
    def y_shapes(self, claimed, vocabulary=None):
        if self.w is None:
            print("choose_w first."); return self
        vocab = vocabulary or list(Y_SHAPES)
        truth = {k: self._ask(extra=Y_SHAPES[k]) for k in vocab}
        print("With |xy| <= %d, the solver says:" % self.N)
        for k in vocab:
            st, wit = truth[k]
            tag = {'sat': 'POSSIBLE', 'unsat': 'impossible',
                   'unknown': 'UNKNOWN'}[st]
            print("   %-10s %-11s %s"
                  % (k, tag, ('e.g. x=%r y=%r z=%r' % wit) if wit else ''))
        if any(st == 'unknown' for st, _ in truth.values()):
            print()
            print("   A shape came back UNKNOWN -- nothing is settled here.")
            return self
        possible = {k for k, (st, _) in truth.items() if st == 'sat'}
        missed, wrong = possible - set(claimed), set(claimed) - possible
        print()
        if not missed and not wrong:
            print("Case analysis COMPLETE: %s."
                  % (', '.join(sorted(possible)) or 'no shape is possible'))
            print("Now discharge it:  refute(K)")
        if missed:
            print("MISSED: %s. Skip a case and the proof has a hole."
                  % ', '.join(sorted(missed)))
        if wrong:
            print("You claimed, the solver refutes: %s."
                  % ', '.join(sorted(wrong)))
        return self

    # -- the discharge ---------------------------------------------------
    def refute(self, K=2):
        # Is there a split surviving EVERY i in 0..K?  unsat => refuted.
        #
        # This is the shape !Cond actually has.  The lemma lets each split
        # choose its OWN i, so a single i is enough to refute but is not
        # required to be -- see pump().
        if self.w is None:
            print("choose_w first."); return self
        st, surv = self._ask(pumps=list(range(K + 1)))
        print("Is there an admissible split that survives every i <= %d?" % K)
        if st == 'unknown':
            print("   UNKNOWN -- the solver gave up. That is NOT a proof.")
            print("   Try a smaller K, or a shorter w.")
        elif st == 'unsat':
            print("   UNSAT -- no split survives.")
            print("   Every admissible split fails for some i <= %d, so" % K)
            print("   !Cond(%s), and %s is NOT regular.  QED."
                  % (self.name, self.name))
            print("   The solver answered 'for ALL splits'. No example was")
            print("   chosen, so no example could be cherry-picked.")
        else:
            x, y, z = surv
            print("   SAT -- this split survives all of i <= %d:" % K)
            print("     x=%r  y=%r  z=%r" % (x, y, z))
            print("   Your proof has a hole. Raise K, or choose a better w.")
        return self

    def pump(self, i=2):
        # Probe a single i.  Sufficient to refute, not necessary.
        if self.w is None:
            print("choose_w first."); return self
        st, surv = self._ask(pumps=[i])
        where = 'down to i=0' if i == 0 else 'up to i=%d' % i
        print("Pumping %s -- does any split survive THIS i?" % where)
        if st == 'unknown':
            print("   UNKNOWN. Not a proof.")
        elif st == 'unsat':
            print("   UNSAT -- this one i breaks every split. Refuted.")
        else:
            print("   SAT -- survives: x=%r y=%r z=%r" % surv)
            print("   Not a failure: another split may need a different i.")
            print("   refute(K) is the honest test.")
        return self

## 3. Tests

**The stock predicate.** Does it decide $0^n1^n$ correctly?

In [ ]:
for w in ['', '01', '0011', '000111', '001', '0101', '10', '0001']:
    s = Solver(); s.add(P_anbn(StringVal(w)))
    print("   %-8r in 0^n 1^n ? %s" % (w, s.check() == sat))
print()
print("Note what the predicate is doing. 0*1* is a regex, which z3 takes")
print("directly -- but '#0 = #1' is NOT regular and no regex can state it.")
print("Counting underneath the shape constraint is what gets past that.")

**Your move.** The adversary picks $N$; you pick $w$.

In [ ]:
game = PumpGame(P_anbn, '0^n 1^n')
game.adversary(4)
game.choose_w('0' * 4 + '1' * 4)

The lemma constrains **members**, and only the long ones. Try to cheat and you are told so.

In [ ]:
PumpGame(P_anbn, '0^n 1^n').adversary(4).choose_w('0001')
print()
PumpGame(P_anbn, '0^n 1^n').adversary(6).choose_w('01')

**All cases of $y$.** Claim the shapes you believe $|xy|\le N$ allows; the solver marks your work.

In [ ]:
game.y_shapes(['all 0s'])

**Why that $w$ was a good choice.** A shorter one leaves three cases instead of one &mdash; three times the work, three chances to skip one.

In [ ]:
sloppy = PumpGame(P_anbn, '0^n 1^n').adversary(4)
sloppy.choose_w('0011')
sloppy.y_shapes(['all 0s'])

**The discharge.** Not an example &mdash; the whole quantifier.

In [ ]:
game.refute(2)

A single $i$ also breaks every split here. That is *sufficient* to refute, but the lemma lets each split choose its own $i$, so it is not *necessary*.

In [ ]:
game.pump(2)
print()
game.pump(0)

**The solver is not an oracle.** Ask something it cannot decide and it says so &mdash; and `unknown` is not a proof.

In [ ]:
hard = PumpGame(zeros_then_ones(lambda a, b: a != b), '0^n 1^m with n != m')
hard.TIMEOUT_MS = 3000
hard.adversary(4)
hard.choose_w('0' * 4 + '1' * 5)
hard.refute(6)
print()
print("That language IS non-regular, and this w is a poor choice for it:")
print("pumping the 0s keeps the two counts unequal, so a split can survive")
print("many i. A w that works needs (#1s - #0s) divisible by every possible")
print("|y| -- the classic N! trick. The solver will not invent it for you.")
print()
print("Chapter 4, Concept 19 made this point without a solver: failing to")
print("falsify proves nothing. Here you can watch it happen.")

## 4. Exercises


1. `P_lt` and `P_2n` are defined above ($0^n1^m$ with $n<m$, and $0^n1^{2n}$).
   Play the game on each. Which $w$ collapses the case analysis to one shape?
2. Write a predicate for $\{0^n 1^n 0^n\}$, using `zeros_then_ones` as a
   model. Does z3 still decide it, or does it go `unknown`?
3. Take a **regular** language &mdash; $0^*1^*$, which is the shape constraint
   with no counting at all. Run `refute(2)` on it. What comes back, and why is
   that the right answer?
4. In `refute(K)` the test is "survives every $i\le K$", not "fails for some
   $i\le K$". Write both in predicate logic and say why only one of them is
   the negation you want.
5. The adversary fixes $N$ before you choose $w$, but after seeing your
   strategy. Where does that ordering appear in the code, and what would break
   if you could choose $w$ first?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4/Concept-Pumping-Lemma-Predicate-Logic')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')